# Running a program with error correction

The same tiny program, three ways: noiseless, noisy, and noisy *with an error
correction scheme applied*. Nothing about the program changes — only the
substrate it runs on.

In [1]:
import qdk
from qdk import qsharp
from qdk.simulation import run_qir

qsharp.init(target_profile=qdk.TargetProfile.Adaptive)
qir = qsharp.compile("""
{
    use q = Qubit();
    X(q);
    MResetZ(q)
}
""")

# At some point we could only run noiseless simulations
run_qir(qir, shots=4, type="clifford")

[One, One, One, One]

Flip a qubit and measure it: the answer is `One`, every shot.

Real hardware is not noiseless, so the next thing we added was a noise model.

In [2]:
# Currently, we can configure noise
from qdk.simulation import NoiseConfig

noise = NoiseConfig()
noise.x.x = 0.4
run_qir(qir, shots=4, type="clifford", noise=noise)

[Zero, One, Zero, One]

With a 40% error rate on `X`, the answers are wrong much of the time, and
nothing in the program can tell which ones.

That is what an error correction scheme fixes. A **qodec** describes one: the
code, and the fault-tolerant circuits ("gadgets") that implement each logical
operation. Pass one to `run_qir` and the program's qubits are encoded into the
code's logical qubits, the encoded circuit is simulated, and the logical
outcomes are decoded back into ordinary results.

In [3]:
# Now we can incorporate an error correction strategy.
import qdk.ec as ec

c4 = ec.load("c4.qodec.yaml")
run_qir(qir, shots=4, type="clifford", noise=noise, qodec=c4)

[Zero, One, One, One]

Two things are different about that result.

The values are **logical** measurements, reconstructed from four physical qubits
rather than read off one. And there may be **fewer than four** of them: `c4` is
the [[4,2,2]] code, which *detects* errors rather than correcting them, so shots
where it caught a fault are discarded rather than reported as if they were
trustworthy.

That trade — some shots discarded, the rest more reliable — is the whole point,
so let's measure it across a range of noise levels.

In [4]:
from qdk.ec.targets import run_qir_encoded

SHOTS = 2000


def error_rate(results):
    """Fraction of shots that did not report the correct answer, `One`."""
    if not results:
        return float("nan")
    return sum(1 for shot in results if str(shot) != "One") / len(results)


print(f"{'gate error':>10} {'physical':>9} {'encoded':>9} {'detected':>9} {'kept':>6}")
for p in (0.01, 0.02, 0.05, 0.1, 0.2, 0.4):
    level = NoiseConfig()
    level.x.x = p

    physical = run_qir(qir, shots=SHOTS, type="clifford", noise=level)
    every = run_qir_encoded(qir, c4, shots=SHOTS, noise=level, postselect=False)
    kept = run_qir_encoded(qir, c4, shots=SHOTS, noise=level, postselect=True)

    print(f"{p:>10.0%} {error_rate(physical):>9.1%} {error_rate(every):>9.1%} "
          f"{error_rate(kept):>9.1%} {len(kept) / SHOTS:>6.0%}")

gate error  physical   encoded  detected   kept
        1%      0.4%      1.6%      0.6%    98%
        2%      1.6%      2.9%      1.2%    96%
        5%      4.9%      6.8%      2.9%    91%
       10%     10.2%     13.2%      6.9%    81%
       20%     22.4%     25.4%     15.2%    70%
       40%     41.3%     40.2%     32.8%    56%


Two lessons in that table.

**Encoding alone does not help.** Spreading one qubit across four gives noise
more places to strike, so the raw encoded error rate (third column) is *worse*
than the bare physical qubit. The code earns its keep only through its checks.

**Error detection does help, and it helps most when noise is low.** At a 1% gate
error the detected-and-kept error rate is roughly half the physical one, at the
cost of discarding a couple of percent of shots. At 40% the code is swamped —
errors are so common that many land in ways the checks cannot see, and most
shots get thrown away for little gain. That is the expected behaviour of a
distance-2 code, and it is exactly why the earlier 4-shot run at 40% looked
unimpressive.

## What a qodec has to provide

A qodec supplies a finite logical instruction set — the operations its author
wrote fault-tolerant gadgets for. A program using anything else cannot be
encoded, and `run_qir` will say so rather than quietly running that operation
unprotected.

In [5]:
from qdk.ec.targets import encodable_gates_of

print("c4 can encode:", sorted(encodable_gates_of(c4)))

h_program = qsharp.compile("""
{
    use q = Qubit();
    H(q);
    MResetZ(q)
}
""")

try:
    run_qir(h_program, shots=4, type="clifford", qodec=c4)
except NotImplementedError as error:
    print("\nrefused:", error)

c4 can encode: ['I', 'M', 'MResetZ', 'MZ', 'X', 'Z']

refused: qodec 'c4' cannot encode QIR gate 'H'; it can express ['I', 'M', 'MResetZ', 'MZ', 'X', 'Z']


## Where to go next

* `qdk.ec` — load, save, and complete qodecs, or synthesize one straight from a
  stabilizer code with `qodec_from_code`.
* `qdk.ec.action`, `.checks`, `.distance`, `qdk.ec.equivalence`, `qdk.ec.lint` —
  characterize a qodec and verify it does what its author intended.
* `qdk.ec.targets` — samplers, detector error models, and circuit-level distance.

`qdk_ec_walkthrough.ipynb` covers the full develop / test / deploy lifecycle, and
`qodec_from_code.ipynb` builds a qodec from nothing but a list of stabilizers.